In [ ]:
import json
import dotenv
dotenv.load_dotenv()

from pydantic import BaseModel
from typing import Literal
from pydantic_ai import Agent

class JudgeEvaluation(BaseModel):
    reasoning: str
    label: Literal["good", "bad"]

judge_instructions = """You are an expert evaluator assessing a recipe assistant that answers
cooking questions using a fixed recipe collection.

A response is "good" if:
1. It accurately answers using ONLY information from the recipe collection
2. It correctly identifies when a recipe is not available and says so
3. It correctly identifies out-of-scope questions (not about cooking/recipes) and declines

A response is "bad" if ANY of these apply:
1. It makes up recipes, ingredients, or instructions not in the collection (hallucination)
2. It provides cooking advice (substitutions, nutrition info, storage tips) that it
   invented rather than found in the recipe data
3. It answers a question it should have declined (not about recipes, or recipe not available)
4. It gives incorrect information from the recipes (wrong times, wrong ingredients, wrong steps)
5. It says it cannot help when a matching recipe exists in the collection — be skeptical
   of responses that claim nothing exists for common cuisines or ingredients
6. It answers about completely different recipes than what was asked. If the user asks
   about specific named dishes (e.g. "sushi vs falafel") and the agent responds about
   entirely different dishes (e.g. desserts) without mentioning the requested dishes,
   that is a wrong-scope retrieval failure and should be labelled "bad". A legitimate
   "not in collection" response must at least acknowledge the specific dishes asked about.""" 

judge_agent = Agent(
    'google-gla:gemini-2.5-flash',
    output_type=JudgeEvaluation,
    instructions=judge_instructions,
)

with open('synthetic_results.json') as f:
    results = json.load(f)

for i, r in enumerate(results):
    prompt = f"""Question: {r['question']}
Agent response: {r['output']}"""
    
    evaluation = await judge_agent.run(prompt)
    r['judge_label'] = evaluation.output.label
    r['judge_reasoning'] = evaluation.output.reasoning
    print(f"[{i+1}/{len(results)}] {r['judge_label']}: {r['question']}")

with open('synthetic_results_judged.json', 'w') as f:
    json.dump(results, f, indent=2)

[1/20] good: What are the main spices in Chicken Tikka Masala?
[2/20] good: How do you marinate the chicken for Chicken Tikka Masala?
[3/20] good: What is the cooking time for the sauce before adding the cream and chicken?
[4/20] good: What can I use instead of heavy cream if I'm dairy-free?
[5/20] good: How does Chicken Tikka Masala differ from Butter Chicken?
[6/20] good: What is the baking temperature for chocolate lava cake?
[7/20] good: How do I achieve the 'lava' effect in the cake?
[8/20] bad: What kind of chocolate is best for this recipe?
[9/20] good: Can I use milk chocolate instead of dark chocolate?
[10/20] good: How does chocolate lava cake compare to a regular chocolate cake?
[11/20] good: What are the main ingredients in Miso Soup?
[12/20] good: How do you make Miso Soup?
[13/20] bad: What kind of tofu is used in Miso Soup?
[14/20] good: Can I use regular miso paste instead of white miso paste?
[15/20] good: How does Miso Soup compare to Tonkotsu Ramen?
[16/20] good: Wha

In [3]:
import json

with open('synthetic_results_judged.json') as f:
    results = json.load(f)

total = len(results)
bad = sum(1 for r in results if r['judge_label'] == 'bad')
print(f"Total: {total}")
print(f"Bad: {bad} ({bad/total*100:.1f}%)")

KeyError: 'judge_label'